Cell 01
# Day 5  California Housing Price Prediction Service
## 전체 파이프라인 개요

Cell 02
## Cell 01  환경 설정: 작업 디렉토리 고정 + 패키지 설치

In [1]:
# Cell 03
import sys, os

# [중요] 모든 상대경로의 기준을 MLOps01으로 고정
os.chdir("/Users/macminim4/Aiffel02/MLOps01")
print(f"작업 디렉토리: {os.getcwd()}")

# venv의 pip 사용 보장 (시스템 pip과 혼동 방지)
!{sys.executable} -m pip install -q torch scikit-learn fastapi nest_asyncio uvicorn requests streamlit
print("패키지 설치 완료")

작업 디렉토리: /Users/macminim4/Aiffel02/MLOps01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Users/macminim4/Aiffel02/MLOps01/.venv/bin/python -m pip install --upgrade pip
패키지 설치 완료


Cell 04
## Section 1  데이터 탐색 (Exploratory Data Analysis, EDA)
### 사용 데이터셋: California Housing

Cell 05
### Cell 02  데이터 로드 및 기본 탐색

In [2]:
# Cell 06
from sklearn.datasets import fetch_california_housing
import pandas as pd

# sklearn 내장 함수로 바로 로드 (파일 다운로드 없음)
data = fetch_california_housing()

print(f"샘플 수: {data.data.shape[0]:,}")   # shape[0] = 행(row) = 샘플 수
print(f"피처 수: {data.data.shape[1]}")     # shape[1] = 열(column) = 피처 수
print(f"피처 목록: {list(data.feature_names)}")
print(f"타겟: {data.target_names}")

# pandas DataFrame으로 변환 → 표 형태로 보기 편함
# data.data = 입력 피처 (NumPy array)
# data.feature_names = 열 이름 리스트
df = pd.DataFrame(data.data, columns=data.feature_names)
df['price'] = data.target  # 타겟 열 추가

print()
print("--- 앞 5행 ---")
print(df.head())   # head() 기본값 = 5행

print()
print("--- 데이터 타입 및 결측값 ---")
print(df.info())

샘플 수: 20,640
피처 수: 8
피처 목록: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
타겟: ['MedHouseVal']

--- 앞 5행 ---
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  

--- 데이터 타입 및 결측값 ---
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   Hou

Cell 07
## Section 2  모델 준비: 학습  전처리  저장

Cell 08
### Cell 03  라이브러리 임포트 및 데이터 준비

In [3]:
# Cell 09
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# 데이터 로드
data = fetch_california_housing()
X, y = data.data, data.target       # X = 피처 행렬, y = 가격 벡터
feature_names = list(data.feature_names)  # 피처 이름 리스트 (나중에 JSON 저장용)

print(f"X shape: {X.shape}")        # (20640, 8) — 20640행 × 8열
print(f"y shape: {y.shape}")        # (20640,)   — 1D 벡터
print(f"가격 범위: {y.min():.2f} ~ {y.max():.2f} ($100,000 단위)")
print()
print("피처별 기초 통계:")
for i, name in enumerate(feature_names):
    print(f"  {name:12s}  평균: {X[:, i].mean():10.2f}  표준편차: {X[:, i].std():10.2f}")

X shape: (20640, 8)
y shape: (20640,)
가격 범위: 0.15 ~ 5.00 ($100,000 단위)

피처별 기초 통계:
  MedInc        평균:       3.87  표준편차:       1.90
  HouseAge      평균:      28.64  표준편차:      12.59
  AveRooms      평균:       5.43  표준편차:       2.47
  AveBedrms     평균:       1.10  표준편차:       0.47
  Population    평균:    1425.48  표준편차:    1132.43
  AveOccup      평균:       3.07  표준편차:      10.39
  Latitude      평균:      35.63  표준편차:       2.14
  Longitude     평균:    -119.57  표준편차:       2.00


Cell 10
### Cell 04  학습/테스트 데이터 분리

In [4]:
# Cell 11
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # *your code* — 테스트 비율 20%
    random_state=42     # *your code* — 재현성을 위한 랜덤 시드 고정
)

print(f"학습 데이터:  {X_train.shape[0]:,}개  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"테스트 데이터: {X_test.shape[0]:,}개  ({X_test.shape[0]/len(X)*100:.0f}%)")

학습 데이터:  16,512개  (80%)
테스트 데이터: 4,128개  (20%)


Cell 12
### Cell 05  전처리: 정규화 (Normalization)

In [5]:
# Cell 13
# 공식: X_norm = (X - mean) / std

# [중요] 학습 데이터로만 mean, std 계산
# axis=0 = 열(피처)별 평균/표준편차
train_mean = X_train.mean(axis=0)  # *your code* — 피처별 평균, shape: (8,)
train_std  = X_train.std(axis=0)   # *your code* — 피처별 표준편차, shape: (8,)

print("피처별 평균:", np.round(train_mean, 2))
print("피처별 표준편차:", np.round(train_std, 2))

# 정규화 적용
# [중요] X_test에도 train_mean, train_std 사용 (테스트 통계 사용 금지!)
X_train_norm = (X_train - train_mean) / train_std  # *your code* — 정규화 공식
X_test_norm  = (X_test  - train_mean) / train_std

print()
print(f"정규화 후 학습 데이터 평균: {X_train_norm.mean(axis=0).round(4)}")  # 거의 0
print(f"정규화 후 학습 데이터 표준편차: {X_train_norm.std(axis=0).round(4)}")  # 거의 1

피처별 평균: [ 3.88000e+00  2.86100e+01  5.44000e+00  1.10000e+00  1.42645e+03
  3.10000e+00  3.56400e+01 -1.19580e+02]
피처별 표준편차: [1.90000e+00 1.26000e+01 2.39000e+00 4.30000e-01 1.13702e+03 1.15800e+01
 2.14000e+00 2.01000e+00]

정규화 후 학습 데이터 평균: [-0. -0.  0. -0. -0. -0.  0. -0.]
정규화 후 학습 데이터 표준편차: [1. 1. 1. 1. 1. 1. 1. 1.]


Cell 14
### Cell 06  NumPy 배열  PyTorch 텐서 변환

In [6]:
# Cell 15
# FloatTensor = 32bit 실수형 텐서 (회귀에 적합)
X_train_tensor = torch.FloatTensor(X_train_norm)
X_test_tensor  = torch.FloatTensor(X_test_norm)

y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)  # *your code* — (N,) → (N,1)
y_test_tensor  = torch.FloatTensor(y_test).unsqueeze(1)

print(f"X_train 텐서: {X_train_tensor.shape}")   # torch.Size([16512, 8])
print(f"y_train 텐서: {y_train_tensor.shape}")   # torch.Size([16512, 1])
print(f"X_test  텐서: {X_test_tensor.shape}")
print(f"y_test  텐서: {y_test_tensor.shape}")

X_train 텐서: torch.Size([16512, 8])
y_train 텐서: torch.Size([16512, 1])
X_test  텐서: torch.Size([4128, 8])
y_test  텐서: torch.Size([4128, 1])


Cell 16
### Cell 07  모델 정의: HousingModel

In [7]:
# Cell 17
class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델 (회귀)"""

    def __init__(self, input_dim=8):
        super().__init__()
        # nn.Sequential: 레이어를 순서대로 쌓는 컨테이너
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),  # *your code* — 입력(8) → 64
            nn.ReLU(),                 # 음수는 0으로 (비선형성 추가)
            nn.Dropout(0.2),           # 20% 뉴런 랜덤 비활성화 → 과적합 방지
            nn.Linear(64, 32),         # 64 → 32 (압축)
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),          # *your code* — 32 → 1 (회귀: 가격 1개 출력)
        )

    def forward(self, x):
        # model(x) 호출 시 자동으로 이 함수가 실행됨
        return self.network(x)

model = HousingModel(input_dim=8)
print(f"모델 구조:")
print(model)
print(f"총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

모델 구조:
HousingModel(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
총 파라미터 수: 2,689


Cell 18
### Cell 08  학습 설정: DataLoader + Loss + Optimizer

In [8]:
# Cell 19
from torch.utils.data import TensorDataset, DataLoader

# TensorDataset: X와 y를 묶어서 한 번에 관리
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# DataLoader: mini-batch 자동 생성기
# batch_size=256: 한 번에 256개 샘플로 가중치 업데이트
# shuffle=True: 매 epoch마다 순서 섞기
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# 손실함수: MSELoss = 회귀의 표준 (분류는 CrossEntropyLoss)
criterion = nn.MSELoss()  # *your code* — 회귀이므로 MSELoss

# 옵티마이저: Adam (학습률 자동 조정)
# lr=1e-3 = 0.001 (딥러닝 표준 기본값)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # *your code*

EPOCHS = 50  # 전체 데이터를 몇 번 반복 학습할지

print(f"배치 수: {len(train_loader)}개/epoch  (16512 ÷ 256 = 65)")
print(f"총 학습 스텝: {len(train_loader) * EPOCHS:,}번")

배치 수: 65개/epoch  (16512 ÷ 256 = 65)
총 학습 스텝: 3,250번


Cell 20
### Cell 09  학습 루프 (Training Loop)

In [9]:
# Cell 21
# model.train(): Dropout 활성화 (학습 모드)
model.train()

for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0

    for X_batch, y_batch in train_loader:  # 매 배치 반복
        # Step 1: 이전 기울기 초기화 (필수! 누적 방지)
        optimizer.zero_grad()

        # Step 2: 순전파 — 모델이 예측값 계산
        predictions = model(X_batch)           # *your code* — 순전파

        # Step 3: 손실 계산 — 예측값과 정답의 차이
        loss = criterion(predictions, y_batch)  # *your code* — MSE 계산

        # Step 4: 역전파 — 각 가중치의 기울기 계산
        loss.backward()

        # Step 5: 가중치 업데이트 — 기울기 방향으로 이동
        optimizer.step()

        running_loss += loss.item()

    # 10 epoch마다 진행상황 출력
    if epoch % 10 == 0:
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch:3d}/{EPOCHS} — Loss: {avg_loss:.4f}")

Epoch  10/50 — Loss: 0.5580
Epoch  20/50 — Loss: 0.4848
Epoch  30/50 — Loss: 0.4549
Epoch  40/50 — Loss: 0.4291
Epoch  50/50 — Loss: 0.4018


Cell 22
### Cell 10  모델 평가

In [10]:
# Cell 23
# Cell 10 — 모델 평가 (테스트 데이터 사용)
# model.eval(): 평가 모드 (Dropout OFF)
model.eval()

with torch.no_grad():  # 기울기 계산 비활성화 → 메모리/속도 절약
    test_preds = model(X_test_tensor)
    test_loss  = criterion(test_preds, y_test_tensor)  # MSE

    # MAE: 평균 절대 오차 (실제 오차 크기를 직접 나타냄)
    mae = torch.abs(test_preds - y_test_tensor).mean().item()

print(f"테스트 MSE: {test_loss.item():.4f}")
print(f"테스트 MAE: {mae:.4f} ($100,000 단위)")
print(f"테스트 MAE: ${mae * 100000:,.0f} (실제 금액 환산)")
print()
print("해석: 평균적으로 실제 가격에서 위 금액만큼 차이남")

테스트 MSE: 0.3347
테스트 MAE: 0.3977 ($100,000 단위)
테스트 MAE: $39,766 (실제 금액 환산)

해석: 평균적으로 실제 가격에서 위 금액만큼 차이남


Cell 24
### Cell 11  모델 및 전처리 파라미터 저장

In [11]:
# Cell 25
# Cell 11 — 모델 및 전처리 파라미터 저장 (배포 시 세트로 저장 필수!)
import os, json

os.makedirs("models", exist_ok=True)

# 1. 모델 가중치 저장 (state_dict = 모든 레이어의 가중치 딕셔너리)
torch.save(model.state_dict(), "models/housing_model.pth")  # *your code*
print(f"모델 저장: models/housing_model.pth  "
      f"({os.path.getsize('models/housing_model.pth')/1024:.1f} KB)")

# 2. 전처리 파라미터 저장 (배포 시 새 데이터 정규화에 필수!)
preprocessing_params = {
    "mean": train_mean.tolist(),    # NumPy → list (JSON 직렬화 가능)
    "std":  train_std.tolist(),
    "feature_names": list(feature_names),
}
with open("models/housing_preprocessing.json", "w") as f:
    json.dump(preprocessing_params, f, indent=2)
print("전처리 파라미터 저장: models/housing_preprocessing.json")

모델 저장: models/housing_model.pth  (13.7 KB)
전처리 파라미터 저장: models/housing_preprocessing.json


Cell 26
### Cell 12  추론 모듈 작성: HousingPredictor

In [12]:
%%writefile /Users/macminim4/Aiffel02/MLOps01/app/housing_model.py
# Cell 27
# Cell 12 — 추론 모듈: HousingModel + HousingPredictor
"""
Day 5 - 주택 가격 예측 모델 정의 + 추론 함수
FastAPI 서버가 이 파일을 import해서 사용합니다.
"""
import json
import torch
import torch.nn as nn
import numpy as np


class HousingModel(nn.Module):
    """캘리포니아 주택 가격 예측 모델 (Cell 07과 동일한 구조)"""
    def __init__(self, input_dim=8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


class HousingPredictor:
    """모델 로드 + 전처리 + 추론을 하나로 캡슐화한 클래스
    
    서버 시작 시 1회 인스턴스 생성 → 요청마다 predict() 호출
    """

    def __init__(self, model_path: str, preprocessing_path: str):
        # 전처리 파라미터 로드 (.json)
        with open(preprocessing_path, "r") as f:
            params = json.load(f)
        self.mean = np.array(params["mean"])            # 정규화에 사용
        self.std  = np.array(params["std"])
        self.feature_names = params["feature_names"]    # 피처 순서 기준

        # 모델 가중치 로드 (.pth)
        self.model = HousingModel(input_dim=len(self.feature_names))
        self.model.load_state_dict(
            torch.load(model_path, map_location="cpu", weights_only=False)
        )
        self.model.eval()  # 평가 모드 고정 (Dropout OFF)

    def predict(self, features: dict) -> dict:
        """피처 딕셔너리를 받아 가격을 예측합니다.
        
        Args:
            features: {"MedInc": 3.5, "HouseAge": 25, ...}
        Returns:
            {"predicted_price": 2.35, "predicted_price_usd": 235000}
        """
        # [중요] 피처를 학습 시와 동일한 순서로 정렬
        # feature_names 순서 기준으로 값 추출
        values = [features[name] for name in self.feature_names]  # *your code*

        # 정규화: 학습 시와 동일한 mean, std 사용
        values = np.array(values, dtype=np.float32)
        normalized = (values - self.mean) / self.std               # *your code*

        # 추론: (1, 8) shape의 텐서로 모델에 입력
        input_tensor = torch.FloatTensor(normalized).unsqueeze(0)  # (1, 8)
        with torch.no_grad():
            output = self.model(input_tensor)

        price = max(output.item(), 0.0)  # 음수 방지 (마지막 레이어에 활성화함수 없어서 가능)
        return {
            "predicted_price":     round(price, 4),
            "predicted_price_usd": int(price * 100000),
        }

Overwriting /Users/macminim4/Aiffel02/MLOps01/app/housing_model.py


Cell 28
### Cell 13  추론 모듈 테스트 (모듈이 정상 작동하는지 확인)

In [13]:
# Cell 29
# Cell 13 — 추론 모듈 테스트
import sys
sys.path.insert(0, "/Users/macminim4/Aiffel02/MLOps01")  # app 패키지 경로 추가

from app.housing_model import HousingPredictor

# HousingPredictor 인스턴스 생성 (모델 + 전처리 파라미터 로드)
predictor = HousingPredictor(
    model_path="models/housing_model.pth",
    preprocessing_path="models/housing_preprocessing.json",
)

# 테스트 데이터의 첫 번째 샘플로 검증
# {피처이름: 값} 딕셔너리 형태로 입력
sample_features = {name: float(X_test[0, i]) for i, name in enumerate(feature_names)}
print(f"입력 피처: {sample_features}")

result = predictor.predict(sample_features)
print(f"\n예측 가격: ${result['predicted_price_usd']:,}")
print(f"실제 가격: ${int(y_test[0] * 100000):,}")
print(f"오차:      ${abs(result['predicted_price_usd'] - int(y_test[0]*100000)):,}")

입력 피처: {'MedInc': 1.6812, 'HouseAge': 25.0, 'AveRooms': 4.192200557103064, 'AveBedrms': 1.0222841225626742, 'Population': 1392.0, 'AveOccup': 3.8774373259052926, 'Latitude': 36.06, 'Longitude': -119.01}

예측 가격: $67,712
실제 가격: $47,700
오차:      $20,012


Cell 30
## Section 3  FastAPI 백엔드: 추론 API 서버 구축
### FastAPI란?
### 전체 요청 흐름

Cell 31
### Cell 14  Pydantic 스키마 정의

In [14]:
%%writefile /Users/macminim4/Aiffel02/MLOps01/app/housing_schemas.py
# Cell 32
# Cell 14 — Pydantic 스키마: 요청/응답 데이터 형식 정의
"""
Day 5 - 주택 가격 예측 API 스키마
Pydantic: 타입 검증 + 자동 문서화 + 직렬화를 한번에 처리
"""
from pydantic import BaseModel, Field


class HousingRequest(BaseModel):
    """주택 가격 예측 요청 스키마 — 입력 검증 자동화"""
    # Field(...): 필수값  |  gt/ge/le: 범위 제한
    MedInc:     float = Field(..., gt=0,               description="중위 소득")           # *your code* — gt=0
    HouseAge:   float = Field(..., ge=0,   le=100,     description="주택 연식 (년)")      # *your code* — ge, le
    AveRooms:   float = Field(..., gt=0,               description="평균 방 수")
    AveBedrms:  float = Field(..., gt=0,               description="평균 침실 수")
    Population: float = Field(..., gt=0,               description="인구")
    AveOccup:   float = Field(..., gt=0,               description="평균 거주 인원")
    Latitude:   float = Field(..., ge=32,  le=42,      description="위도 (캘리포니아)")   # *your code*
    Longitude:  float = Field(..., ge=-125, le=-114,   description="경도 (캘리포니아)")

    # Swagger UI에 표시될 예제 값
    model_config = {
        "json_schema_extra": {
            "examples": [{"MedInc": 3.5, "HouseAge": 25.0, "AveRooms": 5.0,
                          "AveBedrms": 1.0, "Population": 1500.0, "AveOccup": 3.0,
                          "Latitude": 37.5, "Longitude": -122.0}]
        }
    }


class HousingResponse(BaseModel):
    """주택 가격 예측 응답 스키마"""
    success:             bool  = Field(description="요청 처리 성공 여부")
    predicted_price:     float = Field(description="예측 가격 ($100,000 단위)")
    predicted_price_usd: int   = Field(description="예측 가격 (USD)")
    input_features:      dict  = Field(description="입력된 피처 값 (디버깅용)")

Overwriting /Users/macminim4/Aiffel02/MLOps01/app/housing_schemas.py


Cell 33
### Cell 15  FastAPI 서버 구현

In [15]:
%%writefile /Users/macminim4/Aiffel02/MLOps01/app/housing_api.py
# Cell 34
# Cell 15 — FastAPI 서버: 추론 엔드포인트 + 비동기 처리
"""
Day 5 - 주택 가격 예측 FastAPI 서버
핵심: startup 이벤트로 모델 1회 로드, run_in_executor로 비동기 추론
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, HTTPException

from app.housing_schemas import HousingRequest, HousingResponse
from app.housing_model import HousingPredictor
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware

logger = setup_logger("housing_api")

# FastAPI 앱 인스턴스 생성
app = FastAPI(
    title="California Housing Price API",
    description="캘리포니아 주택 가격을 예측하는 API",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)  # 요청 로깅 미들웨어
register_error_handlers(app)                 # 에러 핸들러 등록

# 추론 전용 스레드풀: 동시 최대 4개 추론 처리
inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="housing")  # *your code*

MODEL_PATH      = "models/housing_model.pth"
PREPROCESS_PATH = "models/housing_preprocessing.json"
predictor = None  # 서버 시작 전까지 None


@app.on_event("startup")
async def startup():
    """서버 시작 시 1회 실행 — 모델을 메모리에 로드"""
    global predictor
    logger.info("주택 가격 모델 로드 중...")
    predictor = HousingPredictor(MODEL_PATH, PREPROCESS_PATH)  # *your code*
    logger.info("모델 로드 완료")


@app.get("/health", tags=["System"])
async def health_check():
    """서버 상태 확인 — Streamlit 사이드바에서 주기적으로 호출"""
    return {
        "status": "healthy" if predictor is not None else "loading",
        "model": "California Housing",
    }


@app.post("/predict", response_model=HousingResponse, tags=["Prediction"])
async def predict_housing(request: HousingRequest):
    """주택 정보를 받아 가격을 예측합니다.
    
    Pydantic이 입력값을 자동 검증합니다 (범위 초과 → 422 자동 반환).
    """
    if predictor is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    features = request.model_dump()  # *your code* — {"MedInc": 3.5, ...}

    try:
        # run_in_executor: PyTorch 동기 코드를 별도 스레드에서 실행
        # → FastAPI async 서버가 블로킹되지 않음
        loop = asyncio.get_running_loop()           # Python 3.10+ 권장
        result = await loop.run_in_executor(        # *your code*
            inference_executor,   # 스레드풀
            predictor.predict,    # 실행할 함수
            features,             # 함수 인자
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return HousingResponse(
        success=True,
        predicted_price=result["predicted_price"],
        predicted_price_usd=result["predicted_price_usd"],
        input_features=features,
    )

Overwriting /Users/macminim4/Aiffel02/MLOps01/app/housing_api.py


Cell 35
### Cell 16  서버 실행

In [16]:
# Cell 36
# Cell 16 — FastAPI 서버 실행 (노트북 내 백그라운드 실행)
# [주의] 이전에 실행 중이면 커널을 재시작하세요
import nest_asyncio, uvicorn, threading, time, asyncio

nest_asyncio.apply()  # Jupyter의 event loop와 충돌 방지

def run_server():
    # [핵심 Fix] 스레드 내에 명시적으로 새 event loop 생성
    asyncio.set_event_loop(asyncio.new_event_loop())
    uvicorn.run(
        "app.housing_api:app",  # "파일경로:FastAPI인스턴스변수명"
        host="0.0.0.0",
        port=8000,
        loop="asyncio"          # uvloop 대신 표준 asyncio 사용 (스레드 호환)
    )

# daemon=True: 메인 프로세스(노트북) 종료 시 서버도 자동 종료
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)  # 서버 완전히 뜰 때까지 대기
print("서버 시작: http://localhost:8000")
print("Swagger UI: http://localhost:8000/docs")

INFO:     Started server process [15616]
INFO:     Waiting for application startup.


2026-04-02 19:19:19 INFO     [housing_api] 주택 가격 모델 로드 중...
2026-04-02 19:19:19 INFO     [housing_api] 모델 로드 완료


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


서버 시작: http://localhost:8000
Swagger UI: http://localhost:8000/docs


Cell 37
### Cell 17  헬스체크: 서버 정상 작동 확인

In [17]:
# Cell 38
# Cell 17 — 헬스체크 (서버가 살아있는지 확인)
import requests, json

# GET 요청 — 데이터 없이 상태만 물어봄 (vs POST: 데이터 포함)
resp = requests.get("http://localhost:8000/health")
print(f"상태 코드: {resp.status_code}")   # 200 = 정상
print(f"응답: {resp.json()}")

INFO:     127.0.0.1:60841 - "GET /health HTTP/1.1" 200 OK
상태 코드: 200
응답: {'status': 'healthy', 'model': 'California Housing'}


Cell 39
### Cell 18  추론 테스트: 실제 예측 요청

In [18]:
# Cell 40
# Cell 18 — 추론 테스트: POST /predict 호출
sample_request = {
    "MedInc": 3.5,      "HouseAge": 25.0,
    "AveRooms": 5.0,    "AveBedrms": 1.0,
    "Population": 1500.0, "AveOccup": 3.0,
    "Latitude": 37.5,   "Longitude": -122.0,
}

# POST 요청: JSON body에 데이터를 담아 전송
resp   = requests.post("http://localhost:8000/predict", json=sample_request)
result = resp.json()

print(f"상태 코드: {resp.status_code}")
print(f"예측 가격: ${result['predicted_price_usd']:,}")
print()
print("전체 응답:")
print(json.dumps(result, indent=2, ensure_ascii=False))

INFO:     127.0.0.1:60843 - "POST /predict HTTP/1.1" 200 OK
상태 코드: 200
예측 가격: $179,672

전체 응답:
{
  "success": true,
  "predicted_price": 1.7967,
  "predicted_price_usd": 179672,
  "input_features": {
    "MedInc": 3.5,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1500.0,
    "AveOccup": 3.0,
    "Latitude": 37.5,
    "Longitude": -122.0
  }
}


Cell 41
### Cell 19  에러 테스트: 잘못된 입력 처리 검증
# = sample_request의 모든 키-값 복사 + Latitude만 50.0으로 덮어씀

In [19]:
# Cell 42
# Cell 19 — 에러 테스트: Pydantic 검증이 제대로 작동하는지 확인
# 정상적인 서비스에서 잘못된 입력이 들어와도 서버가 죽지 않아야 함

# 에러 1: 필수 필드 누락 (MedInc 하나만 보냄)
resp = requests.post("http://localhost:8000/predict", json={"MedInc": 3.5})
print(f"필드 누락  (7개 빠짐)  → HTTP {resp.status_code}")  # 422 예상

# 에러 2: 위도 범위 초과 (캘리포니아는 32~42)
# **sample_request: 기존 딕셔너리 복사 + Latitude만 덮어쓰기
resp = requests.post("http://localhost:8000/predict",
                     json={**sample_request, "Latitude": 50.0})
print(f"위도 범위 초과 (50.0)  → HTTP {resp.status_code}")  # 422 예상

# 에러 3: 음수 소득 (gt=0 조건 위반)
bad = {**sample_request, "MedInc": -1.0}
resp = requests.post("http://localhost:8000/predict", json=bad)
print(f"소득 음수 (-1.0)       → HTTP {resp.status_code}")  # 422 예상

POST /predict → 422 (0.001s)


INFO:     127.0.0.1:60845 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict → 422 (0.001s)


필드 누락  (7개 빠짐)  → HTTP 422
INFO:     127.0.0.1:60847 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


POST /predict → 422 (0.0s)


위도 범위 초과 (50.0)  → HTTP 422
INFO:     127.0.0.1:60849 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
소득 음수 (-1.0)       → HTTP 422


Cell 43
## Section 4  Streamlit 프론트엔드: 사용자 인터페이스
### Streamlit이란?
### 핵심 위젯
### st.session_state가 필요한 이유

Cell 44
### Cell 20  Streamlit 대시보드 코드 작성

In [20]:
%%writefile /Users/macminim4/Aiffel02/MLOps01/frontend/app_housing.py
# Cell 45
# Cell 20 — Streamlit 대시보드: 입력 폼 + API 호출 + 결과 표시
"""
Day 5 - 캘리포니아 주택 가격 예측 대시보드
구조: 사이드바(서버상태) + 메인(입력 폼 | 결과 표시)
"""
import streamlit as st
import requests

# 페이지 기본 설정
st.set_page_config(page_title="주택 가격 예측", layout="wide")

API_BASE = "http://localhost:8000"


def call_api(url, json_data=None, method="post"):
    """FastAPI 호출 헬퍼 함수 — 에러를 사용자 친화적으로 처리"""
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()  # *your code* — 4xx/5xx 시 예외 발생
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("서버에 연결할 수 없습니다. FastAPI 서버를 실행하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"서버 에러 (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"오류: {type(e).__name__}")
        return None


# 사이드바: 서버 상태 표시
with st.sidebar:
    st.header("설정")
    # /health 호출로 서버 상태 확인
    health = call_api(f"{API_BASE}/health", method="get")
    if health and health.get("status") == "healthy":
        st.success("서버 연결됨")
        server_ok = True
    else:
        st.error("서버 연결 실패")
        server_ok = False
    st.divider()
    st.caption("California Housing Price Predictor")
    st.caption("Day 5 - 프로젝트 1")


st.title("캘리포니아 주택 가격 예측")
st.write("주택 정보를 입력하면 예상 가격을 예측합니다.")

# 메인 영역: 2컬럼 레이아웃 (입력 | 결과)
col_input, col_result = st.columns(2)

with col_input:
    st.subheader("주택 정보 입력")

    c1, c2 = st.columns(2)
    with c1:
        med_inc    = st.number_input("중위 소득 (MedInc)",      min_value=0.1,  max_value=20.0,    value=3.5,    step=0.1)  # *your code*
    with c2:
        house_age  = st.number_input("주택 연식 (HouseAge)",    min_value=0.0,  max_value=100.0,   value=25.0,   step=1.0)
    c1, c2 = st.columns(2)
    with c1:
        ave_rooms  = st.number_input("평균 방 수 (AveRooms)",   min_value=0.1,  max_value=50.0,    value=5.0,    step=0.1)
    with c2:
        ave_bedrms = st.number_input("평균 침실 수 (AveBedrms)",min_value=0.1,  max_value=20.0,    value=1.0,    step=0.1)
    c1, c2 = st.columns(2)
    with c1:
        population = st.number_input("인구 (Population)",       min_value=1.0,  max_value=50000.0, value=1500.0, step=100.0)
    with c2:
        ave_occup  = st.number_input("평균 거주 인원 (AveOccup)",min_value=0.1,  max_value=20.0,    value=3.0,    step=0.1)
    c1, c2 = st.columns(2)
    with c1:
        latitude   = st.number_input("위도 (Latitude)",         min_value=32.0, max_value=42.0,    value=37.5,   step=0.1)  # *your code*
    with c2:
        longitude  = st.number_input("경도 (Longitude)",        min_value=-125.0,max_value=-114.0, value=-122.0, step=0.1)

with col_result:
    st.subheader("예측 결과")
    if not server_ok:
        st.error("서버에 연결할 수 없습니다.")
    else:
        if st.button("가격 예측", type="primary", use_container_width=True):
            request_data = {                              # *your code*
                "MedInc":      med_inc,
                "HouseAge":    house_age,
                "AveRooms":    ave_rooms,
                "AveBedrms":   ave_bedrms,
                "Population":  population,
                "AveOccup":    ave_occup,
                "Latitude":    latitude,
                "Longitude":   longitude,
            }
            with st.spinner("예측 중..."):
                result = call_api(f"{API_BASE}/predict", json_data=request_data)
            if result:
                # session_state: 페이지 재실행 후에도 결과 유지
                st.session_state["last_housing_result"] = result

        if "last_housing_result" in st.session_state:
            result = st.session_state["last_housing_result"]
            # st.metric: 큰 숫자를 강조해서 표시
            st.metric(label="예상 주택 가격", value=f"${result['predicted_price_usd']:,}")
            st.caption(f"모델 출력값: {result['predicted_price']} ($100,000 단위)")
            with st.expander("입력된 피처 확인"):
                for key, value in result["input_features"].items():
                    st.write(f"**{key}**: {value}")

Overwriting /Users/macminim4/Aiffel02/MLOps01/frontend/app_housing.py


Cell 46
### Cell 21  Streamlit 프론트엔드 실행
# 터미널 1 (이미 실행 중)
# 터미널 2

In [ ]:
# Cell 47
# Cell 21 — Streamlit 프론트엔드 백그라운드 실행
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run",
     "/Users/macminim4/Aiffel02/MLOps01/frontend/app_housing.py",
     "--server.port", "8501"],
    stdout=subprocess.DEVNULL,  # 로그를 노트북에 안 보이게
    stderr=subprocess.DEVNULL,
    cwd="/Users/macminim4/Aiffel02/MLOps01",  # 절대경로로 실행
)
time.sleep(3)
print("프론트엔드 실행: http://localhost:8501")
print()
print("테스트 시나리오:")
print("  1. 기본값 그대로 '가격 예측' 버튼 클릭 → 가격 확인")
print("  2. 중위소득 8.0으로 올리기 → 가격 상승 확인")
print("  3. 중위소득 1.0으로 내리기 → 가격 하락 확인")

Cell 48
## Section 5  통합 테스트: 전체 서비스 검증

Cell 49
### Cell 22  통합 테스트 시작

In [ ]:
# Cell 50
# Cell 22 — 통합 테스트 초기화
# FastAPI 서버가 실행 중이어야 합니다 (Cell 16 실행 후)
import requests, json, time

API_BASE = "http://localhost:8000"
print("=" * 60)
print("  통합 테스트 시작")
print("=" * 60)

Cell 51
### Cell 23  테스트 1: 다양한 입력 검증

In [ ]:
# Cell 52
# Cell 23 — 테스트 1: 다양한 입력으로 모델 동작 확인
# 소득이 높을수록 예측 가격이 높아야 함 (상식적 검증)
test_cases = [
    {"name": "저소득 지역", "MedInc": 1.5,  "HouseAge": 40, "AveRooms": 4.0,
     "AveBedrms": 1.0, "Population": 2000, "AveOccup": 3.5,
     "Latitude": 34.0, "Longitude": -118.0},
    {"name": "고소득 지역", "MedInc": 10.0, "HouseAge": 10, "AveRooms": 8.0,
     "AveBedrms": 2.0, "Population": 500,  "AveOccup": 2.0,
     "Latitude": 37.8, "Longitude": -122.4},
    {"name": "평균적 주택", "MedInc": 3.5,  "HouseAge": 25, "AveRooms": 5.0,
     "AveBedrms": 1.0, "Population": 1500, "AveOccup": 3.0,
     "Latitude": 37.5, "Longitude": -122.0},
]

print("\n[테스트 1] 정상 요청 - 다양한 입력")
print(f"{'케이스':<15} {'예측 가격':>12}")
print("-" * 30)
for case in test_cases:
    name = case.pop("name")
    resp = requests.post(f"{API_BASE}/predict", json=case)   # *your code* — POST 요청
    result = resp.json()
    print(f"{name:<15} ${result['predicted_price_usd']:>10,}")
    case["name"] = name  # 다음 테스트를 위해 name 복원

Cell 53
### Cell 24  테스트 2: 에러 상황 처리 검증

In [ ]:
# Cell 54
# Cell 24 — 테스트 2: 에러 상황에서 서버가 안 죽는지 검증
# 422: 클라이언트 오류 (잘못된 입력) — Pydantic이 자동 처리
# 500: 서버 오류 (코드 버그) — 발생하면 안 됨
print("\n[테스트 2] 에러 상황")

resp = requests.post(f"{API_BASE}/predict", json={"MedInc": 3.5})
print(f"  필드 누락      → HTTP {resp.status_code}")  # 422 예상

bad_req = {**test_cases[2], "Latitude": 50.0}
bad_req.pop("name", None)
resp = requests.post(f"{API_BASE}/predict", json=bad_req)
print(f"  위도 범위 초과  → HTTP {resp.status_code}")  # 422 예상

bad_req2 = {**test_cases[2], "MedInc": -1.0}
bad_req2.pop("name", None)
resp = requests.post(f"{API_BASE}/predict", json=bad_req2)
print(f"  소득 음수      → HTTP {resp.status_code}")  # 422 예상

Cell 55
### Cell 25  테스트 3: 동시 요청 처리 검증

In [ ]:
# Cell 56
# Cell 25 — 테스트 3: 동시 요청 (run_in_executor 효과 확인)
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_predict(i):
    """단일 예측 요청 전송 및 응답 시간 측정"""
    case = test_cases[i % len(test_cases)].copy()
    case.pop("name", None)
    start = time.time()
    resp = requests.post(f"{API_BASE}/predict", json=case, timeout=30)
    return {"id": i+1, "elapsed": round(time.time()-start, 3), "status": resp.status_code}

print("\n[테스트 3] 동시 요청 8개")
start = time.time()
# max_workers=8: 8개 스레드가 동시에 요청 전송
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(send_predict, i) for i in range(8)]
    results = [f.result() for f in as_completed(futures)]

total = round(time.time()-start, 2)
for r in sorted(results, key=lambda x: x["id"]):
    print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
print(f"\n  전체 소요: {total}초 (동시 처리 덕분에 빠름!)")

Cell 57
### Cell 26  테스트 4 + 결과 종합

In [ ]:
# Cell 58
# Cell 26 — 테스트 4: 헬스체크 + 결과 종합
print("\n[테스트 4] 헬스체크")
resp = requests.get(f"{API_BASE}/health")
print(f"  상태: {resp.json()}")

print("\n" + "=" * 60)
print("  통합 테스트 결과 종합")
print("=" * 60)
print("  정상 요청: 다양한 입력에서 합리적인 가격 반환")
print("  에러 처리: 잘못된 입력에 422 반환, 서버 안 죽음")
print("  동시 처리: 8개 동시 요청 정상 처리")
print("  헬스체크: 서버 상태 정상")
print()
print("배포 준비 완료!")